In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T, models



In [2]:
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)
cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# paths
train_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images/"
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"




Using device: cuda


In [3]:
class CassavaDataset(Dataset):
    def __init__(self, img_dir, df=None, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        if df is not None:
            self.images = df["image_id"].tolist()
            self.labels = df["label"].tolist()
        else:
            # test set – no labels
            self.images = sorted(
                [f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")]
            )
            self.labels = None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            label = self.labels[idx]
            return img, label, img_name
        else:
            return img, img_name


# transforms
train_transforms = T.Compose(
    [
        T.RandomResizedCrop(384, scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transforms = T.Compose(
    [
        T.Resize((384, 384)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [4]:
# Load training metadata
train_df = pd.read_csv(train_csv_path)

# Datasets / loaders
batch_size = 64
num_workers = 4

train_dataset = CassavaDataset(train_dir, df=train_df, transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

test_dataset = CassavaDataset(test_dir, df=None, transform=test_transforms)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

# Model: EfficientNet B0 pretrained on ImageNet, adapted to 5 classes
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = torch.nn.Linear(in_features, 5)
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)



Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 104MB/s]


In [5]:
# Simple training loop (few epochs to keep runtime reasonable)
epochs = 3
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for imgs, labels, _ in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * imgs.size(0)
    epoch_loss /= len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}")



Epoch [1/3]  Loss: 0.6260
Epoch [2/3]  Loss: 0.3953
Epoch [3/3]  Loss: 0.3414


In [6]:
# Inference on test set
model.eval()
all_names = []
all_preds = []
softmax = torch.nn.Softmax(dim=1)

with torch.no_grad():
    for imgs, filenames in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = softmax(outputs)
        pred_labels = torch.argmax(probs, dim=1).cpu().tolist()
        all_names.extend(filenames)
        all_preds.extend(pred_labels)

# Create submission file
submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path} – rows:", len(submission))

Submission saved to submission.csv – rows: 2676
